# Order Book Signals on Bond & Equity Calendar Spreads

## 1  Setup & Connection

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '..')

import polars as pl
import polars.selectors as cs
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from plotnine import *
from plotnine.themes import theme_bw

from src.utils import (
    connect_snowflake,
    create_snowpark_session,
    retrieve_polars_from_snowpark,
    unpack_kwargs,
    unpack_kwargs_for_agg,
    read_table,
    parse_security,
    build_contract_calendar,
    add_roll_window,
    add_microstructure_signals,
    acf_by_security,
    ljungbox_by_security,
    newey_west_maxlags,
)
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window
from functools import reduce
import operator

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(20)
print('polars', pl.__version__)

polars 1.41.2


In [2]:
DB     = 'LISTED_INTERN_PROJECT'
SCHEMA = 'PROJECT_5'
TABLES = ['BINNED_DATA', 'QCODE_MAPPING', 'SECURITY_META']

snowflake_conn = connect_snowflake('../.env')
snowpark = create_snowpark_session('../.env')
print('Connected.')

Connected.


## 2  Data Ingestion

Pull only the required columns from each Snowflake table (the projection is pushed down to
Snowflake so we transfer the minimum over the wire). Column names come back lower-cased.

In [3]:
# --- 2  Data Ingestion -------------------------------------------------------------
# Small-scale subset: 2 Phys + 2 Cash qcodes, years 2024-2025.
#
# Futures and calendar spreads of the same product carry DIFFERENT QCODEs in BINNED_DATA,
# so we cannot use a single QCODE filter to capture both instrument types.  Instead:
#   • Futures  — matched by QCODE AND SECURITY NOT LIKE '%/%' (no slash in name).
#   • Spreads  — matched by SECURITY LIKE pattern: {BBG_CODE}%/% {YELLOW_KEY}.
#
# The 2024-2025 date restriction is applied in Section 3 after the contract calendar
# is joined (target_date is delivery-type-aware and not available until that join).
#
# Note: Snowpark's Column.contains() treats its argument as a column identifier, not a
# literal string. All substring checks are expressed as LIKE patterns to avoid this.

BINNED_COLS = [
    'QCODE', 'SECURITY', 'BIN_START_TIME', 'PUBLICATION_DATE',
    'BID_SIZE_START', 'ASK_SIZE_START', 'BID_START', 'ASK_START',
    'VOLUME', 'SIGNED_VOLUME',
]
QCODE_COLS    = ['QCODE', 'BBG_CODE', 'YELLOW_KEY', 'DELIVERY', 'IS_CONVENTION_BUY_NEAR']
SEC_META_COLS = ['SECURITY', 'LAST_TRADE_DATE', 'FIRST_NOTICE_DATE']

qmap     = read_table(snowpark, DB, SCHEMA, 'QCODE_MAPPING', QCODE_COLS)
sec_meta = read_table(snowpark, DB, SCHEMA, 'SECURITY_META', SEC_META_COLS)

# Pick 2 Phys + 2 Cash qcodes for small-scale workshopping.
phys_qcodes = qmap.filter(pl.col('delivery') == 'Phys')['qcode'].unique().to_list()[:2]
cash_qcodes = qmap.filter(pl.col('delivery') == 'Cash')['qcode'].unique().to_list()[:2]
subset_qcodes = phys_qcodes + cash_qcodes
print(f'Phys qcodes : {phys_qcodes}')
print(f'Cash qcodes : {cash_qcodes}')

# BBG_CODE + YELLOW_KEY for the 4 selected products — used to construct the spread filter.
subset_products = (
    qmap.filter(pl.col('qcode').is_in(subset_qcodes))
    .select('bbg_code', 'yellow_key')
    .unique()
)

fqn_binned = f'{DB}.{SCHEMA}.BINNED_DATA'

# Futures: QCODE in our subset AND SECURITY has no slash (LIKE '%/%' negated).
futures_filter = (
    F.col('QCODE').isin(subset_qcodes) &
    ~F.col('SECURITY').like('%/%')
)

# Spreads: SECURITY matches pattern  {BBG_CODE}.../{...} {YELLOW_KEY}.
# Using Snowflake LIKE: '%' = zero-or-more wildcard characters.

spread_clauses = [
    F.col('SECURITY').like(f'{row["bbg_code"]}%/% {row["yellow_key"]}')
    for row in subset_products.iter_rows(named=True)
]
spread_filter = reduce(operator.or_, spread_clauses)

snow_binned = (
    snowpark.table(fqn_binned)
    .select(*BINNED_COLS)
    .filter(futures_filter | spread_filter)
)
binned = retrieve_polars_from_snowpark(snow_binned)

n_fut = binned.filter(~pl.col('security').str.contains('/')).height
n_spr = binned.filter(pl.col('security').str.contains('/')).height
print(f'binned   : {binned.shape}  (futures: {n_fut:,}  |  spreads: {n_spr:,})')
print(f'qmap     : {qmap.shape}')
print(f'sec_meta : {sec_meta.shape}')
binned.head()

Phys qcodes : ['GH', 'GM']
Cash qcodes : ['HN', 'FX']
binned   : (3272775, 10)  (futures: 2,184,028  |  spreads: 1,088,747)
qmap     : (18, 5)
sec_meta : (1106, 3)


qcode,security,bin_start_time,publication_date,bid_size_start,ask_size_start,bid_start,ask_start,volume,signed_volume
str,str,time,date,f64,f64,f64,f64,i32,i32
"""GM""","""OE2025M Comdty""",16:15:00,2025-06-03,1671.0,1943.0,119.21,119.22,6422,359
"""GH""","""DU2025U/2025Z Comdty""",09:00:00,2025-08-08,148.0,179.0,-0.085,-0.08,0,0
"""GH""","""DU2025Z/2026H Comdty""",15:15:00,2025-11-24,7010.0,7821.0,-0.01,0.0,0,0
"""FX""","""VG2025H Index""",17:30:00,2025-01-20,2236.0,2691.0,5184.0,5185.0,27822,4043
"""GH""","""DU2025U Comdty""",14:50:00,2025-07-25,4480.0,248.0,107.075,107.08,1475,-413


## 3  Merge, Parse Tickers, Build the Contract Calendar & Filter to 2024-2025

**Step 1** — merge `BINNED_DATA` with `QCODE_MAPPING` on `QCODE`. Spreads pulled via name
pattern may carry a QCODE outside the futures subset; their qmap columns are left as null and
the authoritative `delivery` comes from the calendar (Step 3).

**Step 2** — parse the `SECURITY` string to flag futures vs. calendar spreads and, for every
spread, construct the explicit **near** and **far** single-future tickers. Each row also gets a
`meta_key`: the near leg for spreads (since `SECURITY_META` has no spread rows), the contract
itself for futures.

**Step 3 & 4** — build a per-future roll calendar from `SECURITY_META`: pick the target date
(`LAST_TRADE_DATE` for `Cash`, `FIRST_NOTICE_DATE` for `Phys`), compute each contract's own
10-business-day roll window, and attach the *previous* chronological contract's window. Join the
calendar onto the data via `meta_key`. The qmap's `delivery` column (null for spreads) is dropped
before this join so the calendar's `delivery` (always resolved from BBG_CODE + YELLOW_KEY) is used.

**Step 5** — derive the trading `date` from `PUBLICATION_DATE` (`BIN_START_TIME` is time-only).

**Step 6** — keep only rows whose near-leg `target_date` falls in **2024 or 2025**. This is the
date-range restriction for the workshop subset.

In [4]:
# --- 3  Merge + parse + calendar + date filter ------------------------------------

# Step 1: merge BINNED_DATA with QCODE_MAPPING on QCODE.
# Spreads pulled via the name-pattern filter may have QCODEs outside subset_qcodes,
# so their bbg_code / yellow_key / delivery / is_convention_buy_near will be null here.
data = binned.join(qmap, on='qcode', how='left')

# Step 2: parse SECURITY -> is_spread, near/far identifiers, meta_key.
data = parse_security(data, col='security')

data.head()

# Steps 3-4: build the per-future roll calendar (target date + own/previous roll windows).
calendar = build_contract_calendar(sec_meta, qmap, roll_days=10)

# Drop qmap's 'delivery' before joining the calendar so the calendar's 'delivery'
# (correctly resolved from BBG_CODE+YELLOW_KEY for every security) is the sole authority.
data = data.drop('delivery')
data = data.join(
    calendar.rename({'security': 'meta_key'}),
    on='meta_key',
    how='left',
)

# Step 5: trading date from PUBLICATION_DATE (BIN_START_TIME is time-only, not a date).
data = data.with_columns(date=pl.col('publication_date').cast(pl.Date))

# Step 6: keep only rows whose near-leg target date is in 2024 or 2025.
# target_date = FIRST_NOTICE_DATE (Phys) or LAST_TRADE_DATE (Cash), already set by the calendar.
data = data.filter(pl.col('publication_date').dt.year().is_in([2024, 2025]))

n_spread = data.filter(pl.col('is_spread')).height
print(f'rows: {data.height:,}  |  spread rows: {n_spread:,}  |  future rows: {data.height - n_spread:,}')
data.select(
    'security', 'is_spread', 'near_identifier', 'far_identifier', 'meta_key',
    'delivery', 'target_date', 'roll_start', 'roll_end', 'prev_roll_start', 'prev_roll_end',
).head()

rows: 632,494  |  spread rows: 210,878  |  future rows: 421,616


security,is_spread,near_identifier,far_identifier,meta_key,delivery,target_date,roll_start,roll_end,prev_roll_start,prev_roll_end
str,bool,str,str,str,str,date,date,date,date,date
"""OE2025M Comdty""",false,"""OE2025M Comdty""",null,"""OE2025M Comdty""","""Phys""",2025-06-06,2025-05-23,2025-06-05,2025-02-20,2025-03-05
"""DU2025U/2025Z Comdty""",true,"""DU2025U Comdty""","""DU2025Z Comdty""","""DU2025U Comdty""","""Phys""",2025-09-08,2025-08-25,2025-09-05,2025-05-23,2025-06-05
"""DU2025Z/2026H Comdty""",true,"""DU2025Z Comdty""","""DU2026H Comdty""","""DU2025Z Comdty""","""Phys""",2025-12-08,2025-11-24,2025-12-05,2025-08-25,2025-09-05
"""VG2025H Index""",false,"""VG2025H Index""",null,"""VG2025H Index""","""Cash""",2025-03-21,2025-03-07,2025-03-20,2024-12-06,2024-12-19
"""DU2025U Comdty""",false,"""DU2025U Comdty""",null,"""DU2025U Comdty""","""Phys""",2025-09-08,2025-08-25,2025-09-05,2025-05-23,2025-06-05


## 4  Roll-Period Filtering → `df_cs` & `df_combined`

**Step 7** — keep only rows whose `date` (from `PUBLICATION_DATE`) falls inside the relevant
roll window:

- **Calendar spreads** — inside their *own* roll period (their near leg's window).
- **Futures** — inside *either* their own roll period *or* their immediately-previous
  chronological contract's roll period.

Outputs:
- `df_cs` — filtered calendar spreads only.
- `df_combined` — filtered calendar spreads **and** filtered futures.

In [5]:
# --- 5  Roll-period filtering -----------------------------------------------------

# A bin sits inside a [start, end] window (inclusive). Null bounds (missing metadata) -> False.
def _in_window(start: str, end: str) -> pl.Expr:
    return pl.col('date').is_between(pl.col(start), pl.col(end), closed='both').fill_null(False)

in_own_roll  = _in_window('roll_start', 'roll_end')
in_prev_roll = _in_window('prev_roll_start', 'prev_roll_end')

# Calendar spreads: own (near-leg) roll period only.
df_cs = data.filter(pl.col('is_spread') & in_own_roll)

# Futures: own roll period OR previous contract's roll period.
df_fut = data.filter(~pl.col('is_spread') & (in_own_roll | in_prev_roll))

# Combined: filtered spreads + filtered futures (aligned schema via vertical concat).
df_combined = pl.concat([df_cs, df_fut], how='vertical')

print(f'df_cs       : {df_cs.shape}')
print(f'df_fut      : {df_fut.shape}')
print(f'df_combined : {df_combined.shape}')
df_cs.head()

df_cs       : (44320, 27)
df_fut      : (88640, 27)
df_combined : (132960, 27)


qcode,security,bin_start_time,publication_date,bid_size_start,ask_size_start,bid_start,ask_start,volume,signed_volume,bbg_code,yellow_key,is_convention_buy_near,is_spread,near_identifier,far_identifier,near_expiry_key,far_expiry_key,meta_key,expiry_key,delivery,target_date,roll_start,roll_end,prev_roll_start,prev_roll_end,date
str,str,time,date,f64,f64,f64,f64,i32,i32,str,str,f64,bool,str,str,i32,i32,str,i32,str,date,date,date,date,date,date
"""GH""","""DU2025Z/2026H Comdty""",15:15:00,2025-11-24,7010.0,7821.0,-0.01,0.0,0,0,"""DU""","""Comdty""",1.0,true,"""DU2025Z Comdty""","""DU2026H Comdty""",202512,202603,"""DU2025Z Comdty""",202512,"""Phys""",2025-12-08,2025-11-24,2025-12-05,2025-08-25,2025-09-05,2025-11-24
"""HN""","""XU2025X/2025Z Index""",11:15:00,2025-11-19,302.0,119.0,-33.0,-32.0,118,2,"""XU""","""Index""",0.0,true,"""XU2025X Index""","""XU2025Z Index""",202511,202512,"""XU2025X Index""",202511,"""Cash""",2025-11-27,2025-11-13,2025-11-26,2025-10-16,2025-10-29,2025-11-19
"""FX""","""VG2025U/2025Z Index""",12:05:00,2025-09-15,52999.0,70016.0,-9.5,-9.25,1433,-1037,"""VG""","""Index""",1.0,true,"""VG2025U Index""","""VG2025Z Index""",202509,202512,"""VG2025U Index""",202509,"""Cash""",2025-09-19,2025-09-05,2025-09-18,2025-06-06,2025-06-19,2025-09-15
"""FX""","""VG2025H/2025M Index""",09:45:00,2025-03-17,29161.0,9848.0,53.25,53.5,1382,1078,"""VG""","""Index""",1.0,true,"""VG2025H Index""","""VG2025M Index""",202503,202506,"""VG2025H Index""",202503,"""Cash""",2025-03-21,2025-03-07,2025-03-20,2024-12-06,2024-12-19,2025-03-17
"""HN""","""XU2025U/2025V Index""",12:45:00,2025-09-23,1516.0,362.0,-8.0,-7.0,39,9,"""XU""","""Index""",0.0,true,"""XU2025U Index""","""XU2025V Index""",202509,202510,"""XU2025U Index""",202509,"""Cash""",2025-09-29,2025-09-15,2025-09-26,2025-08-14,2025-08-27,2025-09-23


## 5  Microstructure Signal Generation (Research Plan §3)

Compute the order-book / flow signals on the filtered calendar-spread set `df_cs`. **CRITICAL —
strictly intraday:** all lag-dependent quantities are partitioned by the trading **session
`[SECURITY, DATE]`** (not just the contract) and ordered by `BIN_START_TIME`, so a lag never
spans an overnight gap, a weekend, or a day boundary. The first bin of each session has null
lags and is dropped. Using the per-bin `*_START` quotes:

| Signal | Definition |
|---|---|
| $P_t$ (`mid_price`) | $(\text{BID\_START} + \text{ASK\_START})/2$ |
| $\Delta P_t$ (`delta_p`) | $P_t - P_{t-1}$ (within session) |
| $OBI_t$ (`obi`) | $\text{BID\_SIZE\_START} - \text{ASK\_SIZE\_START}$ |
| $\Delta L_t^b$ (`delta_lb`) | $Q_t^b-Q_{t-1}^b$ if $P_t^b=P_{t-1}^b$; $\;Q_t^b$ if $P_t^b>P_{t-1}^b$; $\;-Q_{t-1}^b$ if $P_t^b<P_{t-1}^b$ |
| $\Delta L_t^a$ (`delta_la`) | mirror of the bid (an ask *improvement* is a price **decrease**) |
| $OBC_t$ (`ofi`) | $\Delta L_t^b - \Delta L_t^a$  (order-flow imbalance) |
| $STV_t$ (`stv`) | `SIGNED_VOLUME` (dataset); empty bins → 0 |
| $NOI_t$ (`noi`) | $OBC_t - STV_t$ |

A `date` session key is added automatically. Result is saved to **`df_signals`**.

In [6]:
# --- 5  Signal generation (partitioned by SESSION [security, date], ordered by time) ----
df_signals = add_microstructure_signals(df_cs, security_col='security', time_col='bin_start_time')

n_sessions = df_signals.select(['security', 'date']).n_unique()
print(f'df_signals: {df_signals.shape}  ({df_signals["security"].n_unique()} spreads, {n_sessions} sessions)')
df_signals.select(
    'security', 'date', 'bin_start_time', 'mid_price', 'delta_p', 'obi',
    'delta_lb', 'delta_la', 'ofi', 'stv', 'noi',
).head()

df_signals: (43840, 35)  (48 spreads, 480 sessions)


security,date,bin_start_time,mid_price,delta_p,obi,delta_lb,delta_la,ofi,stv,noi
str,date,time,f64,f64,f64,f64,f64,f64,i32,f64
"""DU2024H/2024M Comdty""",2024-02-22,08:15:00,-0.5475,-0.005,-85.0,-130.0,173.0,-303.0,0,-303.0
"""DU2024H/2024M Comdty""",2024-02-22,08:20:00,-0.5475,0.0,847.0,932.0,0.0,932.0,0,932.0
"""DU2024H/2024M Comdty""",2024-02-22,08:25:00,-0.5475,0.0,-81.0,-928.0,0.0,-928.0,0,-928.0
"""DU2024H/2024M Comdty""",2024-02-22,08:30:00,-0.5475,0.0,-81.0,0.0,0.0,0.0,0,0.0
"""DU2024H/2024M Comdty""",2024-02-22,08:35:00,-0.5475,0.0,-81.0,0.0,0.0,0.0,0,0.0


In [7]:
# --- 5b  Normalise signals --------------------------------------------------------
# Normalised values overwrite the raw columns so downstream sections (H1, H2, H3)
# use normalised signals automatically.
#
# All five expressions are evaluated against the INPUT state of df_signals in the
# final with_columns call — Polars never sees intermediate results within the same
# call.  This means pl.col('ofi') in the NOI denominator is the PRE-Z-SCORE raw OFI.
#
# fill_nan(None) converts floating-point 0/0 = NaN to null for inactive bins rather
# than propagating NaN into the regression.

ROLL_WIN = 60   # rolling-window length for OFI z-score (bins, within each session)
SESSION  = ['security', 'date']

# 1. Tick size per BBG_CODE: minimum strictly-positive |delta_p|.
tick_sizes = (
    df_signals
    .filter(pl.col('delta_p').abs() > 10e-6)  # filter out zero and near-zero price changes to avoid noise/tick size confusion)
    .group_by('bbg_code')
    .agg(_tick=pl.col('delta_p').abs().min())
)
df_signals = df_signals.join(tick_sizes, on='bbg_code', how='left')

# Pre-compute rolling OFI mean and std within each session.
# df_signals is already sorted by [security, date, bin_start_time].
df_signals = df_signals.with_columns(
    _ofi_mu=pl.col('ofi').rolling_mean(window_size=ROLL_WIN, min_samples=2).over(SESSION),
    _ofi_sd=pl.col('ofi').rolling_std(window_size=ROLL_WIN, min_samples=2).over(SESSION),
)

# Apply all five normalisations in a single pass.
_vol = pl.col('volume').cast(pl.Float64)

df_signals = df_signals.with_columns(
    # 1. delta_p in ticks (per-BBG_CODE tick size).
    delta_p=(pl.col('delta_p') / pl.col('_tick')).fill_nan(None),
    # 2. OBI as fraction of total quoted size at the touch.
    obi=(pl.col('obi') / (pl.col('bid_size_start') + pl.col('ask_size_start'))).fill_nan(None),
    # 3. STV as fraction of unsigned volume.
    stv=(pl.col('stv').cast(pl.Float64) / _vol).fill_nan(None),
    # 4. OFI as rolling z-score within session.
    ofi=((pl.col('ofi') - pl.col('_ofi_mu')) / pl.col('_ofi_sd')).fill_nan(None),
    # 5. NOI normalised by (unsigned volume + raw |OFI|).
    #    pl.col('ofi') below reads the ORIGINAL raw OFI (input state of this with_columns).
    noi=(pl.col('noi') / (_vol + pl.col('ofi').abs())).fill_nan(None),
).drop(['_tick', '_ofi_mu', '_ofi_sd'])

print(f'df_signals (normalised): {df_signals.shape}')
df_signals.select('security', 'date', 'bin_start_time', 'delta_p', 'obi', 'stv', 'ofi', 'noi').describe()

df_signals (normalised): (43840, 35)


statistic,security,date,bin_start_time,delta_p,obi,stv,ofi,noi
str,str,str,str,f64,f64,f64,f64,f64
"""count""","""43840""","""43840""","""43840""",43840.0,43840.0,34192.0,43321.0,41281.0
"""null_count""","""0""","""0""","""0""",0.0,0.0,9648.0,519.0,2559.0
"""mean""",null,"""2025-01-12 17:37:20.145985""","""12:49:07.992700""",-0.005703,-0.050861,0.125722,-0.001863,-0.017679
"""std""",null,null,null,1.004524,0.55046,0.742794,0.989944,0.763549
"""min""","""DU2024H/2024M Comdty""","""2024-01-16""","""08:15:00""",-21.0,-0.999984,-1.0,-7.559432,-1.0
"""25%""",null,"""2024-07-16""","""10:50:00""",0.0,-0.480597,-0.590473,-0.299995,-0.84
"""50%""",null,"""2025-01-10""","""12:40:00""",0.0,-0.043929,0.209354,0.008728,-0.008547
"""75%""",null,"""2025-06-26""","""14:35:00""",0.0,0.342234,0.936376,0.312096,0.800548
"""max""","""XU2025Z/2026F Index""","""2025-12-29""","""18:00:00""",28.0,0.999932,1.0,7.61324,1.0


## 6  Hypothesis H1 — Autocorrelation & Persistence (Research Plan §5.1)

**H1:** both Signed Trade Volume ($STV_t$) and Net Order Inflow ($NOI_t$) exhibit significant,
slowly-decaying positive autocorrelation.

To keep the analysis **strictly intraday**, the ACF and Ljung-Box Q-test are computed
**per session `[SECURITY, DATE]`** (ordered by `BIN_START_TIME`) and then aggregated —
concatenating across day boundaries would inject spurious overnight/weekend lags. For each
$X \in \{STV_t, NOI_t\}$ we report the cross-sectional **mean ACF** out to lag $L=20$ (with the
$\pm 1.96/\sqrt{\bar n}$ white-noise band) and the **Ljung-Box** Q-test, validating H1 if the
null of no serial correlation is rejected at $p < 0.01$.

In [9]:
# --- 6a  Sample ACF up to lag L = 12 (per SESSION [security, date], then averaged) ----
L = 12
SESSION = ['security', 'date']
acf_results = {}

# Mean session length (used for the approximate white-noise band).
avg_n = df_signals.group_by(SESSION).len()['len'].mean()
band = 1.96 / np.sqrt(avg_n)

for col in ['stv', 'noi', 'obi', 'ofi']:
    lags, mean_acf, per_sec, n_series = acf_by_security(
        df_signals, value_col=col, group_cols=SESSION,
        time_col='bin_start_time', nlags=L, min_obs=50,
    )
    acf_results[col] = mean_acf
    print(f'\n=== ACF of {col.upper()}  ({n_series} sessions, ~{avg_n:,.0f} bins each) ===')
    print(f'    95% white-noise band ~ +/-{band:.3f}')
    for lag, val in zip(lags, mean_acf):
        flag = '*' if abs(val) > band else ' '
        print(f'    lag {lag:>2}: {val:+.4f} {flag}')

acf_table = pl.DataFrame({'lag': list(range(1, L + 1)),
                          'acf_stv': acf_results['stv'],
                          'acf_noi': acf_results['noi'],
                          'acf_obi': acf_results['obi'],
                          'acf_ofi': acf_results['ofi']})
acf_table


=== ACF of STV  (366 sessions, ~91 bins each) ===
    95% white-noise band ~ +/-0.205
    lag  1: +0.1910  
    lag  2: +0.1147  
    lag  3: +0.0873  
    lag  4: +0.0658  
    lag  5: +0.0520  
    lag  6: +0.0374  
    lag  7: +0.0287  
    lag  8: +0.0238  
    lag  9: +0.0114  
    lag 10: -0.0018  
    lag 11: -0.0087  
    lag 12: -0.0208  

=== ACF of NOI  (446 sessions, ~91 bins each) ===
    95% white-noise band ~ +/-0.205
    lag  1: -0.1221  
    lag  2: +0.0139  
    lag  3: +0.0128  
    lag  4: +0.0116  
    lag  5: +0.0015  
    lag  6: +0.0053  
    lag  7: +0.0050  
    lag  8: -0.0038  
    lag  9: -0.0110  
    lag 10: -0.0115  
    lag 11: -0.0040  
    lag 12: -0.0185  

=== ACF of OBI  (480 sessions, ~91 bins each) ===
    95% white-noise band ~ +/-0.205
    lag  1: +0.5228 *
    lag  2: +0.3728 *
    lag  3: +0.2797 *
    lag  4: +0.2240 *
    lag  5: +0.1839  
    lag  6: +0.1499  
    lag  7: +0.1229  
    lag  8: +0.0918  
    lag  9: +0.0699  
    lag 10: +

lag,acf_stv,acf_noi,acf_obi,acf_ofi
i64,f64,f64,f64,f64
1,0.191042,-0.122133,0.52284,0.02482
2,0.114735,0.013879,0.372832,0.014906
3,0.087344,0.012828,0.279715,0.019128
4,0.065753,0.011628,0.224044,0.009562
5,0.05199,0.00147,0.183892,0.003449
6,0.037371,0.005345,0.149853,0.002102
7,0.02871,0.005017,0.122857,-0.004751
8,0.02379,-0.003798,0.091822,-0.008691
9,0.011424,-0.01099,0.069939,-0.011734


In [10]:
# --- 6b  Ljung-Box Q-test (cumulative through lag 20), per SESSION [security, date] ----
ALPHA = 0.01

for col in ['stv', 'noi', 'obi', 'ofi']:
    lb = ljungbox_by_security(
        df_signals, value_col=col, group_cols=SESSION,
        time_col='bin_start_time', lag=L, min_obs=50,
    )
    n_total = lb.height
    n_reject = lb.filter(pl.col('lb_pvalue') < ALPHA).height
    print(f'\n=== Ljung-Box Q-test on {col.upper()} (lag {L}) ===')
    print(f'    sessions tested       : {n_total}')
    print(f'    reject H0 @ p<{ALPHA}    : {n_reject}/{n_total} ({n_reject / n_total:.1%})')
    print(f'    median Q p-value      : {lb["lb_pvalue"].median():.2e}')
    print(f'    median Q-statistic    : {lb["lb_stat"].median():,.1f}')

# --- H1 verdict --------------------------------------------------------------------
print('\n' + '=' * 60)
print('H1 verdict: significant, slowly-decaying POSITIVE autocorrelation')
print('is confirmed when the mean ACF stays positive above the band across')
print(f'lags and the Ljung-Box null is rejected (p<{ALPHA}) for ~all sessions.')


=== Ljung-Box Q-test on STV (lag 12) ===
    sessions tested       : 366
    reject H0 @ p<0.01    : 94/366 (25.7%)
    median Q p-value      : 1.96e-01
    median Q-statistic    : 15.9

=== Ljung-Box Q-test on NOI (lag 12) ===
    sessions tested       : 446
    reject H0 @ p<0.01    : 26/446 (5.8%)
    median Q p-value      : 3.73e-01
    median Q-statistic    : 12.9

=== Ljung-Box Q-test on OBI (lag 12) ===
    sessions tested       : 480
    reject H0 @ p<0.01    : 364/480 (75.8%)
    median Q p-value      : 3.73e-09
    median Q-statistic    : 64.3

=== Ljung-Box Q-test on OFI (lag 12) ===
    sessions tested       : 480
    reject H0 @ p<0.01    : 30/480 (6.2%)
    median Q p-value      : 4.18e-01
    median Q-statistic    : 12.4

H1 verdict: significant, slowly-decaying POSITIVE autocorrelation
is confirmed when the mean ACF stays positive above the band across
lags and the Ljung-Box null is rejected (p<0.01) for ~all sessions.


## 7  Hypothesis H2 — Contemporaneous Price Impact (Research Plan §5.2)

Fit the contemporaneous regression

$$\Delta P_t = \beta_0 + \beta_1 STV_t + \beta_2 NOI_t + \beta_3 OBI_t + \epsilon_t$$

$OFI_t$ (`ofi`) is **omitted** because $NOI_t = OFI_t - STV_t$ makes the three flow metrics
perfectly collinear. The regression is contemporaneous (no lags), so all spreads are pooled;
the data stays sorted by `[security, date, bin_start_time]` so the Newey-West HAC window operates
mostly *within* an intraday session. We use **Newey-West HAC** standard errors with lag truncation
$m = \lfloor 4 (T/100)^{2/9} \rfloor$.

**H2 is validated** if $\beta_1$ (STV) and $\beta_2$ (NOI) are positive and highly significant
($t > 2.5$), while $\beta_3$ (OBI) is near zero / insignificant.

In [11]:
# --- 7a  Fit the contemporaneous model with Newey-West HAC errors -----------------
# y = delta_p ; X = [STV, NOI, OBI] (+ const). OFI/ofi omitted to avoid perfect collinearity.
reg_cols = ['delta_p', 'stv', 'noi', 'obi']
reg_df = df_signals.select(reg_cols).drop_nulls().to_pandas()

T = len(reg_df)                       # total valid observations
m = newey_west_maxlags(T)             # m = floor(4 * (T/100)**(2/9))
print(f'Valid observations T = {T:,}')
print(f'Newey-West lag truncation m = {m}')

y = reg_df['delta_p']
X = sm.add_constant(reg_df[['stv', 'noi', 'obi']])

model_h2 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': m})
print(model_h2.summary())

Valid observations T = 34,192
Newey-West lag truncation m = 14
                            OLS Regression Results                            
Dep. Variable:                delta_p   R-squared:                       0.149
Model:                            OLS   Adj. R-squared:                  0.149
Method:                 Least Squares   F-statistic:                     792.9
Date:                Mon, 08 Jun 2026   Prob (F-statistic):               0.00
Time:                        15:21:47   Log-Likelihood:                -48524.
No. Observations:               34192   AIC:                         9.706e+04
Df Residuals:                   34188   BIC:                         9.709e+04
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

In [12]:
# --- 7b  H2 validation checks ------------------------------------------------------
T_STAT_THR = 2.5   # "highly significant"

params, tvals, pvals = model_h2.params, model_h2.tvalues, model_h2.pvalues

def _report(name, key):
    b, t, p = params[key], tvals[key], pvals[key]
    print(f'  {name:<12} beta={b:+.4e}  t={t:+.2f}  p={p:.2e}')
    return b, t, p

print(f'R-squared = {model_h2.rsquared:.4f}  (HAC, maxlags={m})\n')
print('Coefficients:')
b1, t1, p1 = _report('STV (b1)', 'stv')
b2, t2, p2 = _report('NOI (b2)', 'noi')
b3, t3, p3 = _report('OBI (b3)', 'obi')

stv_ok = (b1 > 0) and (t1 > T_STAT_THR)
noi_ok = (b2 > 0) and (t2 > T_STAT_THR)
obi_negligible = (abs(t3) < T_STAT_THR) or (p3 > 0.05)

print('\n--- H2 validation ---')
print(f'  [{"PASS" if stv_ok else "FAIL"}] b1 (STV) positive & t>2.5 : '
      f'beta>0={b1 > 0}, t={t1:+.2f}')
print(f'  [{"PASS" if noi_ok else "FAIL"}] b2 (NOI) positive & t>2.5 : '
      f'beta>0={b2 > 0}, t={t2:+.2f}')
print(f'  [{"PASS" if obi_negligible else "FAIL"}] b3 (OBI) negligible (|t|<2.5 or p>0.05) : '
      f'|t|={abs(t3):.2f}, p={p3:.2e}')

if stv_ok and noi_ok and obi_negligible:
    print('\n==> H2 CONFIRMED: contemporaneous price impact driven by STV & NOI; OBI negligible.')
else:
    print('\n==> H2 NOT fully supported on this sample (see per-criterion results above).')

R-squared = 0.1495  (HAC, maxlags=14)

Coefficients:
  STV (b1)     beta=+1.8092e-01  t=+20.20  p=9.56e-91
  NOI (b2)     beta=+4.5863e-01  t=+39.41  p=0.00e+00
  OBI (b3)     beta=-5.1893e-01  t=-36.23  p=2.41e-287

--- H2 validation ---
  [PASS] b1 (STV) positive & t>2.5 : beta>0=True, t=+20.20
  [PASS] b2 (NOI) positive & t>2.5 : beta>0=True, t=+39.41
  [FAIL] b3 (OBI) negligible (|t|<2.5 or p>0.05) : |t|=36.23, p=2.41e-287

==> H2 NOT fully supported on this sample (see per-criterion results above).


## 8  Hypothesis H3 — Price Change Prediction (Research Plan §5.3)

Fit the **predictive** regression

$$\Delta P_{t+1} = \theta_0 + \theta_1 NOI_t + \theta_2 OBI_t + \theta_3 STV_t + \eta_{t+1}$$

The target $\Delta P_{t+1}$ is `delta_p` shifted **back one bin within each session
`[SECURITY, DATE]`** (current features predicting the *next intraday* bin); the last bin of every
session has no forward target and is dropped — so a prediction never reaches across an overnight
gap into the next day. As in H2, $OFI_t$ (`ofi`) is omitted (collinear with $NOI$/$STV$), and we
use **Newey-West HAC** errors with $m = \lfloor 4 (T/100)^{2/9} \rfloor$ recomputed on the
re-aligned sample.

**H3 is validated** if $\theta_1$ (NOI) and $\theta_2$ (OBI) are positive and highly significant
($t > 2.5$), while $\theta_3$ (STV) is small and negative (mean-reverting) or insignificant.

In [13]:
# --- 8a  Forward-align the target and fit the predictive model --------------------
# delta_p_fwd_t = delta_p_{t+1}: shift delta_p BACK by 1 within each SESSION [security, date],
# so the target is the NEXT INTRADAY bin and never the first bin of the following session.
df_pred = (
    df_signals
    .sort(['security', 'date', 'bin_start_time'])
    .with_columns(delta_p_fwd=pl.col('delta_p').shift(-1).over(['security', 'date']))
    .drop_nulls(subset=['delta_p_fwd'])   # drops the last bin of each session (no intraday t+1)
)

reg_df = df_pred.select(['delta_p_fwd', 'noi', 'obi', 'stv']).drop_nulls().to_pandas()

T_pred = len(reg_df)                  # re-aligned sample size
m_pred = newey_west_maxlags(T_pred)   # m = floor(4 * (T/100)**(2/9))
print(f'Aligned observations T = {T_pred:,}')
print(f'Newey-West lag truncation m = {m_pred}')

y = reg_df['delta_p_fwd']
X = sm.add_constant(reg_df[['noi', 'obi', 'stv']])   # OFI omitted (collinear)

model_h3 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': m_pred})
print(model_h3.summary())

Aligned observations T = 33,823
Newey-West lag truncation m = 14
                            OLS Regression Results                            
Dep. Variable:            delta_p_fwd   R-squared:                       0.103
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     501.0
Date:                Mon, 08 Jun 2026   Prob (F-statistic):          1.52e-318
Time:                        15:22:35   Log-Likelihood:                -50133.
No. Observations:               33823   AIC:                         1.003e+05
Df Residuals:                   33819   BIC:                         1.003e+05
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

In [14]:
# --- 8b  H3 validation checks ------------------------------------------------------
T_STAT_THR = 2.5

params, tvals, pvals = model_h3.params, model_h3.tvalues, model_h3.pvalues

def _report(name, key):
    b, t, p = params[key], tvals[key], pvals[key]
    print(f'  {name:<12} theta={b:+.4e}  t={t:+.2f}  p={p:.2e}')
    return b, t, p

print(f'R-squared = {model_h3.rsquared:.4f}  (HAC, maxlags={m_pred})\n')
print('Coefficients:')
th1, tt1, tp1 = _report('NOI (th1)', 'noi')
th2, tt2, tp2 = _report('OBI (th2)', 'obi')
th3, tt3, tp3 = _report('STV (th3)', 'stv')

noi_ok = (th1 > 0) and (tt1 > T_STAT_THR)
obi_ok = (th2 > 0) and (tt2 > T_STAT_THR)
# STV: small & negative (mean-reverting) OR statistically insignificant.
stv_meanrevert = (th3 < 0) or (abs(tt3) < T_STAT_THR) or (tp3 > 0.05)

print('\n--- H3 validation ---')
print(f'  [{"PASS" if noi_ok else "FAIL"}] th1 (NOI) positive & t>2.5      : '
      f'theta>0={th1 > 0}, t={tt1:+.2f}')
print(f'  [{"PASS" if obi_ok else "FAIL"}] th2 (OBI) positive & t>2.5      : '
      f'theta>0={th2 > 0}, t={tt2:+.2f}')
print(f'  [{"PASS" if stv_meanrevert else "FAIL"}] th3 (STV) negative or insignificant: '
      f'theta={th3:+.2e}, t={tt3:+.2f}, p={tp3:.2e}')

if noi_ok and obi_ok and stv_meanrevert:
    print('\n==> H3 CONFIRMED: NOI & OBI predict next-bin returns; STV is weak/mean-reverting.')
else:
    print('\n==> H3 NOT fully supported on this sample (see per-criterion results above).')

R-squared = 0.1028  (HAC, maxlags=14)

Coefficients:
  NOI (th1)    theta=-1.0210e-01  t=-12.96  p=2.07e-38
  OBI (th2)    theta=+4.8018e-01  t=+33.13  p=1.19e-240
  STV (th3)    theta=+2.5181e-01  t=+23.62  p=2.26e-123

--- H3 validation ---
  [FAIL] th1 (NOI) positive & t>2.5      : theta>0=False, t=-12.96
  [PASS] th2 (OBI) positive & t>2.5      : theta>0=True, t=+33.13
  [FAIL] th3 (STV) negative or insignificant: theta=+2.52e-01, t=+23.62, p=2.26e-123

==> H3 NOT fully supported on this sample (see per-criterion results above).
